# stoX — TFT Training on Colab GPU

**Before running anything:**
1. Go to **Runtime → Change runtime type → T4 GPU** (free tier) or A100 (Colab Pro)
2. **Rebuild the feature panel locally first** (see warning in Step 3)
3. Upload the rebuilt `sl20_feature_panel.parquet` to your Google Drive
4. Run cells top to bottom — each section has a ✅ / ❌ checkpoint

When training finishes the checkpoint is saved to your Drive automatically.

## Step 0 — Reset environment (run this first, every time)

Fixes `getcwd: cannot access parent directories` from stale sessions.

In [ ]:
import os, sys, shutil

os.chdir("/content")
print(f"Working directory: {os.getcwd()}")

REPO_DIR        = "/content/stoX_repo"
DRIVE_MODEL_DIR = "/content/drive/MyDrive/stoX_models/tft_v1"
print(f"REPO_DIR        = {REPO_DIR}")
print(f"DRIVE_MODEL_DIR = {DRIVE_MODEL_DIR}")

## Step 1 — Verify GPU

In [ ]:
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\u26a0\ufe0f  No GPU. Go to Runtime \u2192 Change runtime type \u2192 T4 GPU")

## Step 2 — Clone the stoX repo (master branch)

In [ ]:
GITHUB_USER = "Dineth-San"
GITHUB_REPO = "stoX"
BRANCH      = "master"   # all code lives on master

clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

# Remove stale directory that isn't a real git repo
if os.path.exists(REPO_DIR) and not os.path.exists(f"{REPO_DIR}/.git"):
    shutil.rmtree(REPO_DIR)

if os.path.exists(f"{REPO_DIR}/.git"):
    print("Repo found — pulling latest master...")
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout master
    !git -C {REPO_DIR} pull origin master
else:
    print(f"Cloning {BRANCH} branch...")
    !git clone --branch {BRANCH} {clone_url} {REPO_DIR}

# Show which commit we're on
!git -C {REPO_DIR} log --oneline -3

# Verify key files
checks = [
    f"{REPO_DIR}/ml/pyproject.toml",
    f"{REPO_DIR}/ml/train_model.py",
    f"{REPO_DIR}/ml/configs/pipeline.yaml",
    f"{REPO_DIR}/ml/src/sl20_ml/__init__.py",
]
all_ok = True
for f in checks:
    ok = os.path.exists(f)
    print(f"  {'\u2705' if ok else '\u274c'}  {f.replace(REPO_DIR, '')}")
    if not ok: all_ok = False

# Sanity-check that we have the v2 training changes (FP16 + new arch + new splits)
train_py = open(f"{REPO_DIR}/ml/train_model.py").read()
pipeline_yaml = open(f"{REPO_DIR}/ml/configs/pipeline.yaml").read()
v2_checks = {
    "FP32 precision (no FP16 crash)": '"32-true"' in train_py,
    "Conformal calibration":          "conformal_delta" in train_py,
    "hidden_size=128":                "hidden_size:               128" in pipeline_yaml,
    "encoder_length=90":              "encoder_length:            90" in pipeline_yaml,
    "train_end=2020":                 '"2020-12-31"' in pipeline_yaml,
    "val_start=2021":                 '"2021-01-01"' in pipeline_yaml,
    "learning_rate=1e-3":             "1.0e-3" in pipeline_yaml,
}
print()
for desc, ok in v2_checks.items():
    print(f"  {'✅' if ok else '❌'}  {desc}")
    if not ok: all_ok = False

print("
✅ Repo ready" if all_ok else "
❌ Some v2 checks failed — delete /content/stoX_repo and re-run this cell")

## Step 3 — Mount Google Drive and copy the feature panel

> ⚠️  **The feature panel must be rebuilt locally before uploading.**
>
> The model was upgraded: new splits (train→2020, val→2021) and 5 new features
> (`vol_regime`, `daily_range_pct`, `market_breadth`, `usd_lkr_5d_change`, `foreign_net_flow_30d`).
> The panel from the previous run does **not** have these.
>
> **On your local machine, run:**
> ```powershell
> cd D:\stox\ml
> python build_alignment.py    # rebuilds split labels
> python build_features.py     # adds 5 new features
> ```
> Then upload `ml/data/features/sl20_feature_panel.parquet` to Google Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ── UPDATE THIS to match where you uploaded the file in Drive ─────────────────
PARQUET_IN_DRIVE = "/content/drive/MyDrive/sl20_feature_panel.parquet"
# ──────────────────────────────────────────────────────────────────────────────────

if not os.path.exists(PARQUET_IN_DRIVE):
    print(f"❌ Not found: {PARQUET_IN_DRIVE}")
    print("Check the path — right-click the file in Drive to confirm its location")
else:
    import pandas as pd
    DEST = f"{REPO_DIR}/ml/data/features/sl20_feature_panel.parquet"
    os.makedirs(os.path.dirname(DEST), exist_ok=True)
    shutil.copy(PARQUET_IN_DRIVE, DEST)
    size_mb = os.path.getsize(DEST) / 1e6
    print(f"✅ Panel copied: {size_mb:.1f} MB → {DEST}")

    # Verify the panel has the new v2 features (fail loudly if old panel uploaded)
    df = pd.read_parquet(DEST)
    new_features = ["vol_regime", "daily_range_pct", "market_breadth",
                    "usd_lkr_5d_change", "foreign_net_flow_30d"]
    missing = [f for f in new_features if f not in df.columns]
    if missing:
        print(f"❌ Panel is MISSING new v2 features: {missing}")
        print("   ↳ You uploaded the OLD panel. Rebuild locally first:")
        print("       python build_alignment.py")
        print("       python build_features.py")
        print("   Then re-upload and re-run this cell. Do NOT continue to Step 4.")
    else:
        print(f"✅ Panel has all {len(new_features)} new v2 features")
        if "split" in df.columns:
            train_max = df[df["split"]=="train"]["date"].max()
            val_min   = df[df["split"]=="val"]["date"].min()
            print(f"✅ Train ends: {train_max}  |  Val starts: {val_min}")
        print(f"   Rows: {len(df):,}  |  Columns: {df.shape[1]}  |  Tickers: {df['ticker'].nunique()}")

## Step 4 — Install dependencies

In [ ]:
os.chdir("/content")

!pip install -q "pytorch-forecasting>=1.0" "lightning>=2.0" "mlflow>=2.14" "pandera>=0.20" "pyyaml>=6.0"
os.system(f'pip install -q -e "{REPO_DIR}/ml"')

SRC_DIR = f"{REPO_DIR}/ml/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

ok = True
for mod in ["sl20_ml", "pytorch_forecasting", "lightning", "mlflow"]:
    try:
        m = __import__(mod)
        print(f"  \u2705  {mod} {getattr(m, '__version__', 'ok')}")
    except ImportError as e:
        print(f"  \u274c  {mod}: {e}")
        ok = False

print("\n\u2705 Ready" if ok else "\n\u274c Some imports failed — re-run Steps 2 and 4")

## Step 5 — Route model output to Google Drive

In [ ]:
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

LOCAL_MODEL_DIR = f"{REPO_DIR}/ml/models/tft_v1"
if os.path.islink(LOCAL_MODEL_DIR):   os.unlink(LOCAL_MODEL_DIR)
elif os.path.isdir(LOCAL_MODEL_DIR):  shutil.rmtree(LOCAL_MODEL_DIR)

os.makedirs(f"{REPO_DIR}/ml/models", exist_ok=True)
os.symlink(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)
print(f"✅ Checkpoints → {DRIVE_MODEL_DIR}")

# Remove old .ckpt files from Drive so Lightning saves the new run as
# "best.ckpt" (not "best-v1.ckpt", "best-v2.ckpt", etc.).
# model_config.json is kept — it records the last known metrics.
import glob
old_ckpts = glob.glob(f"{DRIVE_MODEL_DIR}/*.ckpt")
if old_ckpts:
    for f in old_ckpts:
        os.remove(f)
        print(f"  Removed old checkpoint: {os.path.basename(f)}")
    print(f"✅ Cleared {len(old_ckpts)} old checkpoint(s) — new run will save as best.ckpt")
else:
    print("  No old checkpoints to clear")

## Step 6 — Train

Expected: **~40–70 min** on T4 (`lr=1e-3`, should now train 60–120 epochs before early stopping).
Watch for best val_loss improving past epoch 10 — that confirms LR is right.

In [ ]:
import importlib

# Flush any cached sl20_ml modules so git-pulled changes are always loaded fresh.
# Without this, %run reuses the in-memory import cache and ignores new code.
stale = [k for k in sys.modules if k.startswith("sl20_ml")]
for k in stale:
    del sys.modules[k]
importlib.invalidate_caches()
if stale:
    print(f"Flushed {len(stale)} cached sl20_ml module(s) — will reload from disk")
else:
    print("No cached sl20_ml modules to flush")

SRC_DIR = f"{REPO_DIR}/ml/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.chdir(f"{REPO_DIR}/ml")
print(f"CWD: {os.getcwd()}")

%run {REPO_DIR}/ml/train_model.py

## Step 7 — Verify results

In [ ]:
import json
from pathlib import Path

model_dir   = Path(DRIVE_MODEL_DIR)
config_file = model_dir / "model_config.json"

print("Files on Drive:")
for f in sorted(model_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.2f} MB)")

if config_file.exists():
    cfg_out = json.loads(config_file.read_text())
    delta   = cfg_out.get("conformal_delta", 0.0)
    target  = cfg_out.get("target_coverage", 0.80)
    print(f"
✅ Training complete!")
    print(f"  Conformal delta = {delta:+.6f} (target {target:.0%} coverage)")
    print()
    if "val_metrics_raw" in cfg_out:
        v_raw, v = cfg_out["val_metrics_raw"], cfg_out["val_metrics"]
        t_raw, t = cfg_out["test_metrics_raw"], cfg_out["test_metrics"]
        print(f"  Val  raw QC = {v_raw['quantile_coverage']:.1%}  →  calibrated QC = {v['quantile_coverage']:.1%}")
        print(f"       MAE={v['mae']:.4f}  DA={v['directional_accuracy']:.1%}")
        print(f"  Test raw QC = {t_raw['quantile_coverage']:.1%}  →  calibrated QC = {t['quantile_coverage']:.1%}")
        print(f"       MAE={t['mae']:.4f}  DA={t['directional_accuracy']:.1%}")
    else:
        v = cfg_out["val_metrics"]; t = cfg_out["test_metrics"]
        print(f"  Val  → MAE={v['mae']:.4f}  DA={v['directional_accuracy']:.1%}  QC={v['quantile_coverage']:.1%}")
        print(f"  Test → MAE={t['mae']:.4f}  DA={t['directional_accuracy']:.1%}  QC={t['quantile_coverage']:.1%}")
else:
    print("⚠️  model_config.json not found — check Step 6 output for errors")

## Step 8 — Download checkpoint

**Option A (recommended):** [drive.google.com](https://drive.google.com) → `My Drive / stoX_models / tft_v1 /` → download `best.ckpt` + `model_config.json`

**Option B:** Direct browser download:

In [ ]:
from google.colab import files

ckpt = next(model_dir.glob("*.ckpt"), None)
if ckpt:
    files.download(str(ckpt))
    print(f"Downloading: {ckpt.name}")
else:
    print("No .ckpt file found")

if config_file.exists():
    files.download(str(config_file))
    print("Downloading: model_config.json")

## Step 9 — Use the model locally

Put both files in `D:\stox\ml\models	ft_v1\` then:
```powershell
cd D:\stox\ml
python predict.py
```

---

## Troubleshooting

| Error | Fix |
|-------|-----|
| `getcwd` error | Run **Step 0** first |
| ❌ on file checks in Step 2 | Re-run Step 2 — clone may have failed |
| ❌ on v2 checks in Step 2 | Delete repo: `shutil.rmtree('/content/stoX_repo')` then re-run Step 2 |
| `ModuleNotFoundError: sl20_ml` | Re-run Step 2 (all ✅?), then re-run Step 4 |
| Panel missing new features (Step 3) | Rebuild locally: `build_alignment.py` then `build_features.py`, re-upload |
| `CUDA out of memory` | Reduce `batch_size` to 128 in `ml/configs/pipeline.yaml`, re-run Step 6 |
| Drive path not found | Right-click file in Drive → 'Get link' to confirm exact path |
| Session disconnected | Re-run Step 0 → 2 → 4 → 5 → 6. Checkpoint already saved on Drive. |
| QC still low after training | Check Step 3 output — panel must show all 5 new features ✅ |